# NB-03 — Cenários Regulatórios BNDES
## Análise Prescritiva — BYD Camacari 2025-2027

**Autor:** Hermes Agent (Nous Research)  
**Data:** 2026-07-26  
**Linguagem:** Português (pt-br)

---

### Agenda

1. [Definição dos 4 Cenários Regulatórios BNDES](#cenarios)
2. [Métricas Esperadas (Valor Esperado)](#metricas)
3. [Kill Switches e Gatilhos de Risco](#killswitch)
4. [Regras de Decisão por Status](#decisoes)
5. [Validação e Exportação JSON](#validacao)

In [1]:
# -*- coding: utf-8 -*-
"""
NB-03 — Cenários Regulatórios BNDES
Análise Prescritiva — BYD Camacari 2025-2027
"""

import json
import os
from pathlib import Path

# Paleta de cores — brand tokens BYD
COLOR_DARK   = "#0d1117"
COLOR_BLUE   = "#4f8ef7"
COLOR_AMBER  = "#e8a23c"
COLOR_GREEN  = "#34d399"
COLOR_RED    = "#f85149"

BASE_DIR  = Path("__file__").parent if "__file__" in dir() else Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("📁 Diretório de saída:", OUTPUT_DIR)

📁 Diretório de saída: C:\Users\mathe\code_space\orchestration\value-factory\case-studies\byd-camacari-2025-2027\analise-prescritiva\notebooks!\02-canonicos\nb-03-regulatory-scenarios\outputs


<a id='cenarios'></a>
## 1. Definição dos 4 Cenários Regulatórios BNDES

Os cenários são derivados de **D3-INTERDEPENDENCY-S7-ESG.md** e **D3-INTERDEPENDENCY-S3-S4.md**:

| Cenário | Status | Prob. | ViE | Cobertura | Kill Switch | NPV Incentivo | Custo Atraso |
|---------|--------|-------|-----|-----------|-------------|---------------|-------------|
| **A** — Expansão GREEN | 🟢 GREEN | 75% | 22% | 90% | Não | R$ 1,5B | R$ 0 |
| **B** — Tensão AMBER | 🟡 AMBER | 15% | 10% | 50% | Não | R$ 0,8B | R$ 50M (3 meses) |
| **C** — Crise RED | 🔴 RED | 8% | 5% | 18% | **SIM** | R$ 0 (kill) | R$ 200M (6 meses) |
| **D** — Rollback Total | 🔴 RED | 2% | 0% | 0% | **SIM** | R$ 0 | R$ 500M (freeze) |

In [2]:
scenarios = [
    {
        "id": "A",
        "nome": "Expansão GREEN",
        "status": "GREEN",
        "prob": 0.75,
        "vie": 0.22,
        "incentive_coverage": 0.90,
        "BNDES_kill_switch": False,
        "NPV_incentive_B": 1.5,
        "cost_delay_B": 0,
        "delay_months": 0,
        "decision": "Aprovar expansão, hedge 30%",
        "kill_trigger": None,
    },
    {
        "id": "B",
        "nome": "Tensão AMBER",
        "status": "AMBER",
        "prob": 0.15,
        "vie": 0.10,
        "incentive_coverage": 0.50,
        "BNDES_kill_switch": False,
        "NPV_incentive_B": 0.8,
        "cost_delay_B": 0.05,
        "delay_months": 3,
        "decision": "Expansão condicional, hedge 60%, preparar contingência",
        "kill_trigger": None,
    },
    {
        "id": "C",
        "nome": "Crise RED (kill switch)",
        "status": "RED",
        "prob": 0.08,
        "vie": 0.05,
        "incentive_coverage": 0.18,
        "BNDES_kill_switch": True,
        "NPV_incentive_B": 0.0,
        "cost_delay_B": 0.20,
        "delay_months": 6,
        "decision": "Congelar capex não-essencial, hedge 90%, ativar protocolo de crise",
        "kill_trigger": (
            "Lista suja MTE ativa desde 07/abr/2026 — "
            "163 trabalhadores resgatados dez/2024 — "
            "R$ 800M em financiamento bloqueado"
        ),
    },
    {
        "id": "D",
        "nome": "Rollback Total",
        "status": "RED",
        "prob": 0.02,
        "vie": 0.00,
        "incentive_coverage": 0.00,
        "BNDES_kill_switch": True,
        "NPV_incentive_B": 0.0,
        "cost_delay_B": 0.50,
        "delay_months": 12,
        "decision": "Freeze total, invocar cláusula kill switch",
        "kill_trigger": (
            "Programa integralmente bloqueado — "
            "todas as tranches BNDES suspensas"
        ),
    },
]

print(f"✅ {len(scenarios)} cenários regulatórios definidos com sucesso.")
for s in scenarios:
    icon = {"GREEN": "🟢", "AMBER": "🟡", "RED": "🔴"}.get(s["status"], "⚪")
    print(f"  [{s['id']}] {icon} {s['nome']} — Prob: {s['prob']*100:.0f}% | ViE: {s['vie']*100:.0f}%")

✅ 4 cenários regulatórios definidos com sucesso.
  [A] 🟢 Expansão GREEN — Prob: 75% | ViE: 22%
  [B] 🟡 Tensão AMBER — Prob: 15% | ViE: 10%
  [C] 🔴 Crise RED (kill switch) — Prob: 8% | ViE: 5%
  [D] 🔴 Rollback Total — Prob: 2% | ViE: 0%


<a id='metricas'></a>
## 2. Métricas Esperadas (Valor Esperado)

### Fórmulas

$$
\text{ViE esperado} = \sum_{i} p_i \times \text{ViE}_i
\quad
\text{Cobertura esperada} = \sum_{i} p_i \times \text{cobertura}_i
\quad
\text{NPV esperado} = \sum_{i} p_i \times \text{NPV}_i
$$

### Break-even ViE = 10% (de D3-INTERDEPENDENCY-S3-S4.md)

- **Acima de 10%:** defensivo catalog-wide torna-se viável (ROI positivo)
- **Abaixo de 10%:** defensivo catalog-wide torna-se destrutivo de valor (ROI negativo)

In [3]:
# Cálculos de valor esperado
expected_ViE               = sum(s["prob"] * s["vie"]               for s in scenarios)
expected_incentive_coverage = sum(s["prob"] * s["incentive_coverage"] for s in scenarios)
expected_NPV_B             = sum(s["prob"] * s["NPV_incentive_B"]   for s in scenarios)
kill_switch_prob          = sum(s["prob"] for s in scenarios if s["BNDES_kill_switch"])
break_even_ViE             = 0.10

print("=" * 70)
print("MÉTRICAS ESPERADAS (VALOR ESPERADO):")
print("=" * 70)
print(f"  ViE esperado            : {expected_ViE*100:.1f}%  "
      f"(break-even: {break_even_ViE*100:.0f}%)")
print(f"  Cobertura incentivos    : {expected_incentive_coverage*100:.1f}%")
print(f"  NPV esperado            : R$ {expected_NPV_B:.3f}B")
print(f"  Prob. kill switch       : {kill_switch_prob*100:.0f}%  (cenários C + D)")
print()

# Validação — valores matematicamente corretos
# ViE: 0.75×22 + 0.15×10 + 0.08×5 + 0.02×0 = 18.4%
# Cobertura: 0.75×0.90 + 0.15×0.50 + 0.08×0.18 + 0.02×0.00 = 0.7644
# NPV: 0.75×1.5 + 0.15×0.8 + 0.08×0 + 0.02×0 = 1.245
# Kill switch: 0.08+0.02 = 0.10
assert abs(expected_ViE - 0.184) < 1e-6,               "ViE esperado divergente"
assert abs(expected_incentive_coverage - 0.7644) < 1e-4, "Cobertura esperada divergente"
assert abs(expected_NPV_B - 1.245) < 1e-6,             "NPV esperado divergente"
assert abs(kill_switch_prob - 0.10) < 1e-6,             "Prob. kill switch divergente"
print("✅ Validação OK — todos os valores esperados conferem.")

MÉTRICAS ESPERADAS (VALOR ESPERADO):
  ViE esperado            : 18.4%  (break-even: 10%)
  Cobertura incentivos    : 76.4%
  NPV esperado            : R$ 1.245B
  Prob. kill switch       : 10%  (cenários C + D)

✅ Validação OK — todos os valores esperados conferem.


<a id='killswitch'></a>
## 3. Kill Switches e Gatilhos de Risco

### Cenário C — Crise RED

**Gatilho ativo desde 07/abr/2026:**

- Lista suja MTE (Ministério do Trabalho e Emprego) está **ativa**
- 163 trabalhadores foram resgatados em dez/2024
- R$ 800M em financiamento BNDES estão **bloqueados**
- Cobertura de incentivos cai para 18%
- Kill switch: **TRIGGERS** → NPV = R$ 0, custo de atraso = R$ 200M

### Cenário D — Rollback Total

- Programa integralmente bloqueado
- Todas as tranches BNDES suspensas
- ViE = 0%, cobertura = 0%
- Kill switch: **INVOCADO** → freeze total, custo = R$ 500M

In [4]:
print("=" * 70)
print("KILL SWITCHES — DETALHAMENTO:")
print("=" * 70)

for s in scenarios:
    if s["BNDES_kill_switch"]:
        icon = "🔴"
        print(f"\n  [{s['id']}] {icon} {s['nome']} — KILL SWITCH ATIVO")
        print(f"      Probabilidade : {s['prob']*100:.0f}%")
        print(f"      ViE           : {s['vie']*100:.0f}%")
        print(f"      Cobertura     : {s['incentive_coverage']*100:.0f}%")
        print(f"      NPV incentivo  : R$ {s['NPV_incentive_B']:.1f}B")
        print(f"      Custo atraso   : R$ {s['cost_delay_B']:.2f}B ({s['delay_months']} meses)")
        if s["kill_trigger"]:
            print(f"      Gatilho        : {s['kill_trigger']}")
        print(f"      Decisão        : {s['decision']}")

print()
print(f"⚠️  Probabilidade agregada de kill switch: {kill_switch_prob*100:.0f}%")
print(f"   (Soma dos cenários C + D: 8% + 2%)")

KILL SWITCHES — DETALHAMENTO:

  [C] 🔴 Crise RED (kill switch) — KILL SWITCH ATIVO
      Probabilidade : 8%
      ViE           : 5%
      Cobertura     : 18%
      NPV incentivo  : R$ 0.0B
      Custo atraso   : R$ 0.20B (6 meses)
      Gatilho        : Lista suja MTE ativa desde 07/abr/2026 — 163 trabalhadores resgatados dez/2024 — R$ 800M em financiamento bloqueado
      Decisão        : Congelar capex não-essencial, hedge 90%, ativar protocolo de crise

  [D] 🔴 Rollback Total — KILL SWITCH ATIVO
      Probabilidade : 2%
      ViE           : 0%
      Cobertura     : 0%
      NPV incentivo  : R$ 0.0B
      Custo atraso   : R$ 0.50B (12 meses)
      Gatilho        : Programa integralmente bloqueado — todas as tranches BNDES suspensas
      Decisão        : Freeze total, invocar cláusula kill switch

⚠️  Probabilidade agregada de kill switch: 10%
   (Soma dos cenários C + D: 8% + 2%)


<a id='decisoes'></a>
## 4. Regras de Decisão por Status

| Status | Condição | Ação |
|--------|----------|------|
| 🟢 **GREEN** | ViE ≥ 10%, kill switch OFF | Aprovar expansão, hedge 30% |
| 🟡 **AMBER** | ViE = 10%, kill switch OFF | Expansão condicional, hedge 60%, preparar contingência |
| 🔴 **RED** | ViE < 10% ou kill switch ON | Congelar capex não-essencial, hedge 90%, protocolo de crise |
| 🔴 **ROLLBACK** | Kill switch INVOCADO | Freeze total, invocar cláusula kill switch |

### Break-even ViE = 10%

O break-even de 10% define o limiar entre estratégia **defensiva viável** e **destrutiva de valor**.

In [5]:
print("=" * 70)
print("REGRAS DE DECISÃO POR STATUS:")
print("=" * 70)

decision_rules = {
    "GREEN": (
        "✅ APPROVE: Aprovar expansão, hedge 30%\n"
        "   ViE acima do break-even; incentivos BNDES ativos."
    ),
    "AMBER": (
        "⚠️  CONDITIONAL: Expansão condicional, hedge 60%, preparar contingência\n"
        "   ViE no break-even; risco moderado de revisão."
    ),
    "RED": (
        "🔴 FREEZE: Congelar capex não-essencial, hedge 90%, protocolo de crise\n"
        "   ViE abaixo do break-even OU kill switch ativo."
    ),
}

for status, rule in decision_rules.items():
    icon = {"GREEN": "🟢", "AMBER": "🟡", "RED": "🔴"}.get(status, "⚪")
    print(f"\n  {icon} {status}")
    print(f"   {rule}")

print()
print(f"📌 Break-even ViE: {break_even_ViE*100:.0f}% — "
      f"ViE esperado da carteira: {expected_ViE*100:.1f}%")
if expected_ViE >= break_even_ViE:
    print("   -> ViE esperado está ACIMA do break-even — portfólio VIÁVEL.")
else:
    print("   -> ViE esperado está ABAIXO do break-even — portfólio DEVE SER REVISADO.")

REGRAS DE DECISÃO POR STATUS:

  🟢 GREEN
   ✅ APPROVE: Aprovar expansão, hedge 30%
   ViE acima do break-even; incentivos BNDES ativos.

  🟡 AMBER
   ⚠️  CONDITIONAL: Expansão condicional, hedge 60%, preparar contingência
   ViE no break-even; risco moderado de revisão.

  🔴 RED
   🔴 FREEZE: Congelar capex não-essencial, hedge 90%, protocolo de crise
   ViE abaixo do break-even OU kill switch ativo.

📌 Break-even ViE: 10% — ViE esperado da carteira: 18.4%
   -> ViE esperado está ACIMA do break-even — portfólio VIÁVEL.


<a id='validacao'></a>
## 5. Validação e Exportação JSON

Exporta o resultado estruturado para `outputs/nb03_results.json` com:

- `scenarios`: lista com os 4 cenários completos
- `expected_ViE`: 0.184 (18.4%)
- `expected_incentive_coverage`: 0.7644 (76.44%)
- `expected_NPV_B`: 1.245 (R$ 1.245B)
- `kill_switch_prob`: 0.10 (10%)
- `break_even_ViE`: 0.10 (10%)

In [6]:
results = {
    "scenarios": scenarios,
    "expected_ViE":                round(expected_ViE,               4),
    "expected_incentive_coverage": round(expected_incentive_coverage, 4),
    "expected_NPV_B":             round(expected_NPV_B,              4),
    "kill_switch_prob":            round(kill_switch_prob,           4),
    "break_even_ViE":              break_even_ViE,
    "metadata": {
        "analise":   "Análise Prescritiva — Cenários Regulatórios BNDES",
        "projeto":   "BYD Camacari 2025-2027",
        "data":      "2026-07-26",
        "notebook":   "nb-03-regulatory-scenarios.ipynb",
    },
}

OUTPUT_JSON = OUTPUT_DIR / "nb03_results.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("=" * 70)
print("VALIDAÇÃO E EXPORTAÇÃO JSON:")
print("=" * 70)
print(f"\n💾 JSON gerado em: {OUTPUT_JSON}")
print(f"   Tamanho: {OUTPUT_JSON.stat().st_size} bytes")
print()
print("Conteúdo do JSON:")
print("-" * 70)
print(json.dumps(results, ensure_ascii=False, indent=2))
print("-" * 70)
print()
print("✅ NB-03 executado com sucesso!")
print("=" * 70)

VALIDAÇÃO E EXPORTAÇÃO JSON:

💾 JSON gerado em: C:\Users\mathe\code_space\orchestration\value-factory\case-studies\byd-camacari-2025-2027\analise-prescritiva\notebooks!\02-canonicos\nb-03-regulatory-scenarios\outputs\nb03_results.json
   Tamanho: 2127 bytes

Conteúdo do JSON:
----------------------------------------------------------------------
{
  "scenarios": [
    {
      "id": "A",
      "nome": "Expansão GREEN",
      "status": "GREEN",
      "prob": 0.75,
      "vie": 0.22,
      "incentive_coverage": 0.9,
      "BNDES_kill_switch": false,
      "NPV_incentive_B": 1.5,
      "cost_delay_B": 0,
      "delay_months": 0,
      "decision": "Aprovar expansão, hedge 30%",
      "kill_trigger": null
    },
    {
      "id": "B",
      "nome": "Tensão AMBER",
      "status": "AMBER",
      "prob": 0.15,
      "vie": 0.1,
      "incentive_coverage": 0.5,
      "BNDES_kill_switch": false,
      "NPV_incentive_B": 0.8,
      "cost_delay_B": 0.05,
      "delay_months": 3,
      "decision": 

In [7]:
# Resumo final — comparação visual dos cenários
import json

print("\n" + "=" * 70)
print("RESUMO — COMPARATIVO DE CENÁRIOS:")
print("=" * 70)
print()
print(f"{'Cenário':<30} {'Prob':>6} {'ViE':>6} {'Cobert':>8} {'NPV(B)':>8} {'Kill':>6}")
print("-" * 70)
for s in scenarios:
    print(
        f"{s['nome']:<30} "
        f"{s['prob']*100:>5.0f}% "
        f"{s['vie']*100:>5.0f}% "
        f"{s['incentive_coverage']*100:>7.0f}% "
        f"{s['NPV_incentive_B']:>7.1f} "
        f"{'SIM' if s['BNDES_kill_switch'] else 'NÃO':>5}"
    )
print("-" * 70)
print(f"{'ESPERADO (média ponderada)':<30} "
      f"{'':<6} "
      f"{expected_ViE*100:>5.1f}% "
      f"{expected_incentive_coverage*100:>7.1f}% "
      f"{expected_NPV_B:>7.3f} "
      f"{kill_switch_prob*100:>5.1f}%")
print("=" * 70)


RESUMO — COMPARATIVO DE CENÁRIOS:

Cenário                          Prob    ViE   Cobert   NPV(B)   Kill
----------------------------------------------------------------------
Expansão GREEN                    75%    22%      90%     1.5   NÃO
Tensão AMBER                      15%    10%      50%     0.8   NÃO
Crise RED (kill switch)            8%     5%      18%     0.0   SIM
Rollback Total                     2%     0%       0%     0.0   SIM
----------------------------------------------------------------------
ESPERADO (média ponderada)             18.4%    76.4%   1.245  10.0%
